# Intermediate: Softmax and Temperature — From Scratch

**Sub-topic 1.4 — Math and Statistics Essentials**

An LLM's "next token" step produces one raw score (logit) per possible token in its vocabulary. Softmax turns those raw scores into a valid probability distribution. Temperature reshapes that distribution *before* softmax is applied, controlling how "sharp" (deterministic) or "flat" (random) the sampling is. This notebook makes both operations concrete on a tiny 5-token toy vocabulary.

In [1]:
import numpy as np

# A toy vocabulary and made-up raw scores ("logits") a model might produce
# after seeing the prompt "The sky is". Higher score = model thinks this
# token is more likely to come next.
VOCAB = ["blue", "clear", "cloudy", "purple", "banana"]
LOGITS = np.array([4.2, 3.8, 3.1, 0.5, -2.0])

print(f"Raw logits for vocabulary {VOCAB}: {LOGITS}")

Raw logits for vocabulary ['blue', 'clear', 'cloudy', 'purple', 'banana']: [ 4.2  3.8  3.1  0.5 -2. ]


These raw numbers aren't probabilities yet — they can be negative, and they don't sum to 1. Softmax fixes both problems.

In [2]:
def softmax(logits: np.ndarray) -> np.ndarray:
    """Standard softmax, with the numerically-stable trick of subtracting
    the max logit first (prevents overflow in np.exp for large logits;
    does not change the resulting probabilities -- softmax is shift-invariant)."""
    shifted = logits - np.max(logits)
    exp_scores = np.exp(shifted)
    return exp_scores / exp_scores.sum()


def softmax_with_temperature(logits: np.ndarray, temperature: float) -> np.ndarray:
    """Divide logits by temperature before softmax.
    - temperature < 1: sharpens the distribution (more deterministic)
    - temperature = 1: unchanged (standard softmax)
    - temperature > 1: flattens the distribution (more random)
    """
    return softmax(logits / temperature)


def print_distribution(vocab: list[str], probs: np.ndarray, label: str) -> None:
    print(f"\n{label}")
    for token, p in sorted(zip(vocab, probs), key=lambda x: -x[1]):
        bar = "#" * int(p * 50)
        print(f"  {token:8s} {p:6.1%}  {bar}")
    print(f"  (sum of probabilities = {probs.sum():.4f}, always 1.0 -- that's what makes it a valid distribution)")

In [3]:
base_probs = softmax(LOGITS)
print_distribution(VOCAB, base_probs, "Standard softmax (temperature = 1.0):")


Standard softmax (temperature = 1.0):
  blue      49.3%  ########################
  clear     33.0%  ################
  cloudy    16.4%  ########
  purple     1.2%  
  banana     0.1%  
  (sum of probabilities = 1.0000, always 1.0 -- that's what makes it a valid distribution)


## Now the same logits at three different temperatures

In [4]:
for temperature in [0.3, 1.0, 2.5]:
    probs = softmax_with_temperature(LOGITS, temperature)
    print_distribution(VOCAB, probs, f"Temperature = {temperature}:")


Temperature = 0.3:
  blue      77.6%  ######################################
  clear     20.4%  ##########
  cloudy     2.0%  
  purple     0.0%  
  banana     0.0%  
  (sum of probabilities = 1.0000, always 1.0 -- that's what makes it a valid distribution)

Temperature = 1.0:
  blue      49.3%  ########################
  clear     33.0%  ################
  cloudy    16.4%  ########
  purple     1.2%  
  banana     0.1%  
  (sum of probabilities = 1.0000, always 1.0 -- that's what makes it a valid distribution)

Temperature = 2.5:
  blue      35.6%  #################
  clear     30.4%  ###############
  cloudy    22.9%  ###########
  purple     8.1%  ####
  banana     3.0%  #
  (sum of probabilities = 1.0000, always 1.0 -- that's what makes it a valid distribution)


## Interpretation

At temperature=0.3, `blue` dominates almost completely — sampling from this distribution will pick `blue` nearly every time. This is what a **low** temperature setting in a chat app does: safer, more repeatable output.

At temperature=2.5, the distribution is much flatter — even `banana` (a nonsensical continuation) gets a non-trivial chance of being sampled. This is what a **high** temperature setting does: more variety, but more risk of an incoherent or off-topic output.

Real LLMs run this exact computation over 50,000-200,000+ vocabulary tokens at every single generation step.

### Try it yourself
Try temperature values of `0.1`, `1.0`, and `5.0` on the same logits and describe in your own words what changes about the resulting distribution at each extreme.